In [1]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library.",
    candidate_labels = ["education", "politics", "business"]
)

/Users/paulsundstrom/MLprojects/huggingface-nlp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 5567.63it/s]


{'sequence': 'This is a course about the Transformers library.',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8719870448112488, 0.09406593441963196, 0.03394703194499016]}

In [2]:
from datasets import load_dataset
eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")
eli5 = eli5.train_test_split(test_size= 0.2)

In [4]:
eli5["train"][0]

{'q_id': '5q199q',
 'title': 'What are the benefits of consuming alkaline (pH > 7) food and drink? Does it have any merit?',
 'selftext': 'A friend of mine recently shared a video of a woman and her children testing the pH of a variety of bottled water brands. Any time the pH was above 7, that bottle "passed the test." A pH below 7 was met with disdain. Why?',
 'category': 'Biology',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dcvi8km', 'dcviigq', 'dcvn068'],
  'text': ['Nothing. It\'s total nonsense. The person that popularised the "diet" was recently arrested for practicing medicine without a license. Two people died under his care. It\'s a total scam. You can read about Robert Young here. Including the charges brought against him. He\'s just been sentenced to prison btw. URL_0',
   "My friend has serious acid reflux issues. The doctor recommended alkaline water because it helps neutralize the acid a bit. S'all.",
   "you pad the pockets of the business owner. you creat

In [3]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

In [4]:
eli5 = eli5.flatten()

In [5]:
def preprossess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

In [6]:
tokenized_eli5 = eli5.map(
    preprossess_function,
    batched = True,
    num_proc= 4,
    remove_columns= eli5["train"].column_names
)

Map (num_proc=4):  25%|██▌       | 1000/4000 [00:00<00:01, 1863.72 examples/s][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1308 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1477 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1801 > 1024). Running this sequence through the model will result in indexing errors
Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2362 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than t

In [7]:
block_size = 128

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [8]:
lm_dataset = tokenized_eli5.map(group_texts, batched = True, num_proc= 4)

Map (num_proc=4): 100%|██████████| 1000/1000 [00:00<00:00, 4917.31 examples/s]


In [9]:
from transformers import DataCollatorForLanguageModeling
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer= tokenizer, mlm= False)

In [10]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7461.25it/s]


In [12]:
training_args = TrainingArguments(
    output_dir = "eli_clm_model",
    eval_strategy= "epoch",
    learning_rate= 2e-5,
    weight_decay= 0.01,
    push_to_hub= False,
    num_train_epochs= 1
)

trainer = Trainer(
    model = model,
    args= training_args,
    train_dataset= lm_dataset["train"],
    eval_dataset= lm_dataset["test"],
    data_collator= data_collator,
    processing_class= tokenizer,
)

trainer.train()

/Users/paulsundstrom/MLprojects/huggingface-nlp/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.914824,3.818403


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.64it/s]
/Users/paulsundstrom/MLprojects/huggingface-nlp/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=1323, training_loss=3.902805181317319, metrics={'train_runtime': 776.4472, 'train_samples_per_second': 13.631, 'train_steps_per_second': 1.704, 'total_flos': 345695601033216.0, 'train_loss': 3.902805181317319, 'epoch': 1.0})

In [14]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}" )

Training Loss,Validation Loss,Epoch
3.914824,3.818403,1


Perplexity: 45.53


In [16]:
trainer.save_model("eli_clm_model")

Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.49s/it]


In [26]:
from transformers import AutoTokenizer

prompt = "Somatic hypermutation allows the immunse system to"

tokenizer = AutoTokenizer.from_pretrained("eli_clm_model")
inputs = tokenizer(prompt, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[   50, 13730,  8718,    76,  7094,  3578,   262, 16217,   325,  1080,
           284]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("eli_clm_model")
outputs = model.generate(**inputs, max_new_tokens= 100, do_sample= True, top_k= 50, top_p = 0.95)
tokenizer.batch_decode(outputs, skip_special_tokens= True)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 10336.16it/s]


['Somatic hypermutation allows the immunse system to generate the virus that needs to be present at the time. This gives the virus a chance to pass the immunse to infection without having to get the virus through, thus forcing it to be present in the correct sequence in order to pass it to the target. The main thing to keep in mind is the immunse is a pretty special antibody. The anti-sensitivity test is the measure of how long the vaccine has been effective, and how much it can be removed before it can be fully absorbed']